# Clase 186 — CUPED, sequential testing y always-valid p-values

Tres trucos modernos de A/B testing industrial: **CUPED** reduce varianza usando pre-experiment data (~50% menos sample). **Sequential testing** te deja peekear sin inflar el error tipo I.
Requiere: `pip install numpy scipy`.

## 🧠 Intuición previa

**CUPED en una frase:** usar una covariable *pre-experimento* (algo que ya sabías de cada usuario antes de tratarlo) para **reducir la varianza** de la métrica y así **detectar efectos más chicos con menos muestra**. Si esa covariable correlaciona `ρ` con la métrica, la varianza cae a `(1-ρ²)` — con `ρ=0.7` ya recortás ~la mitad del ruido, y **sin sesgar** el efecto (la covariable es previa; el tratamiento no la afecta).

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
n = 10_000
# Pre-experiment covariate (X), correlación ~0.7 con outcome
X_ctrl = rng.normal(10, 3, n)
X_trt  = rng.normal(10, 3, n)
noise  = lambda k: rng.normal(0, 2.1, k)  # ruido idiosincratico
Y_ctrl = X_ctrl + noise(n)             # mean ~ 10
Y_trt  = X_trt + noise(n) + 0.5        # treatment effect = 0.5
print(f'mean ctrl = {Y_ctrl.mean():.3f}  mean trt = {Y_trt.mean():.3f}  diff = {Y_trt.mean()-Y_ctrl.mean():.3f}')

## t-test naive

In [ ]:
t, p = stats.ttest_ind(Y_trt, Y_ctrl)
var_naive = (Y_trt.var(ddof=1) + Y_ctrl.var(ddof=1)) / 2
print(f'Naive: t={t:.3f}  p={p:.4f}  pooled var ~ {var_naive:.3f}')

## CUPED — Controlled-experiment Using Pre-Experiment Data
$Y^{cuped}_i = Y_i - \theta (X_i - \bar X)$ con $\theta = \dfrac{\text{Cov}(Y,X)}{\text{Var}(X)}$.
$\text{Var}(Y^{cuped}) = \text{Var}(Y)(1 - \rho^2)$ — con $\rho=0.7$, reduce ~50% varianza.

In [ ]:
# Estimamos theta sobre TODO el pool (no por grupo) para no introducir bias
Y_all = np.concatenate([Y_ctrl, Y_trt])
X_all = np.concatenate([X_ctrl, X_trt])
theta = np.cov(Y_all, X_all, ddof=1)[0, 1] / X_all.var(ddof=1)
X_bar = X_all.mean()
Yc_cuped = Y_ctrl - theta * (X_ctrl - X_bar)
Yt_cuped = Y_trt  - theta * (X_trt  - X_bar)

t_c, p_c = stats.ttest_ind(Yt_cuped, Yc_cuped)
var_cuped = (Yt_cuped.var(ddof=1) + Yc_cuped.var(ddof=1)) / 2
print(f'theta = {theta:.3f}')
print(f'CUPED: t={t_c:.3f}  p={p_c:.4e}  pooled var ~ {var_cuped:.3f}')
print(f'Reduccion de varianza: {1 - var_cuped/var_naive:.1%}  (esperado ~50% con rho=0.7)')

## El peligro del *peeking* sin corrección
Simulamos 1000 experimentos NULOS (sin efecto), peekeando 10 veces. Sin corrección, P(rechazar H0) >> 5%.

In [ ]:
rng2 = np.random.default_rng(42)
n_sims = 1000
n_max = 2000
n_peeks = 10
peek_at = np.linspace(200, n_max, n_peeks).astype(int)

rejects_naive = 0
for _ in range(n_sims):
    a = rng2.normal(0, 1, n_max)
    b = rng2.normal(0, 1, n_max)  # SIN efecto
    for k in peek_at:
        _, p = stats.ttest_ind(a[:k], b[:k])
        if p < 0.05:
            rejects_naive += 1
            break
print(f'Type-I error peekeando sin corregir: {rejects_naive/n_sims:.3f}  (debería ser 0.05)')

## Always-valid p-values — corrección Bonferroni para peeks
Si peekeás K veces, usá $\alpha/K$. Conservador pero válido.

In [ ]:
alpha_corr = 0.05 / n_peeks
rng3 = np.random.default_rng(42)
rejects_corr = 0
for _ in range(n_sims):
    a = rng3.normal(0, 1, n_max)
    b = rng3.normal(0, 1, n_max)
    for k in peek_at:
        _, p = stats.ttest_ind(a[:k], b[:k])
        if p < alpha_corr:
            rejects_corr += 1
            break
print(f'Type-I con Bonferroni K={n_peeks} (alpha={alpha_corr:.4f}): {rejects_corr/n_sims:.3f}')

## mSPRT-like: Mixture Sequential Probability Ratio Test
La idea: bajo H0, el likelihood ratio mezclado contra un prior $N(0,\tau^2)$ es martingala. P-value siempre válido en cualquier momento.

In [ ]:
def msprt_pvalue(diffs, sigma, tau=0.1):
    """Always-valid p-value para diferencia de medias bajo H0=0.
    diffs: muestras Y_trt - Y_ctrl pareadas (o pseudo-pareadas por bloque).
    sigma: SD conocida o estimada.
    tau: SD del prior sobre el efecto."""
    n = len(diffs)
    s = diffs.sum()
    # log Bayes factor mezclado
    var_tot = sigma**2 + n*tau**2
    log_bf = 0.5*np.log(sigma**2 / var_tot) + 0.5 * (s*tau)**2 / (sigma**2 * var_tot)
    bf = np.exp(log_bf)
    return min(1.0, 1.0/bf)

rng4 = np.random.default_rng(42)
# Bajo H0: 1000 sims, peekear 10 veces, contar rechazos
rej_msprt = 0
for _ in range(n_sims):
    d = rng4.normal(0, 1, n_max)
    for k in peek_at:
        p = msprt_pvalue(d[:k], sigma=1.0, tau=0.2)
        if p < 0.05:
            rej_msprt += 1
            break
print(f'Type-I con mSPRT (peekeando libre): {rej_msprt/n_sims:.3f}  (target <=0.05)')

## Comparativo final

In [ ]:
print('Tipo I esperado: 0.05')
print(f'Naive peeking      : {rejects_naive/n_sims:.3f}  (inflado)')
print(f'Bonferroni K-peeks : {rejects_corr/n_sims:.3f}  (controlado, conservador)')
print(f'mSPRT always-valid : {rej_msprt/n_sims:.3f}    (controlado, less conservative)')

## Takeaways
1. **CUPED** reduce ~50% varianza con $\rho=0.7$ → mismos resultados con la mitad del tráfico.
2. Peekear sin corrección **infla el type-I error** del 5% al ~20-30%.
3. Bonferroni sobre K peeks es válido pero conservador.
4. **mSPRT / always-valid p-values** te dejan monitorear continuamente sin inflar error — preferido en industria (Optimizely, Netflix).

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables** de los ejercicios del README (sección `## 🧪 Ejercicios`). Datos sintéticos con `np.random.default_rng(42)`, sin internet. Cada bloque imprime resultados y valida con `assert`.

### Ejercicio 1 — CUPED implementation
`Y_post = 0.8·X_pre + tratamiento + eps`. Estimar θ y comparar `Var(Y)` vs `Var(Y_cuped)`.

In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(42)
n = 4000
X_pre = rng.normal(0, 1, n)
treat = rng.integers(0, 2, n)
eps = rng.normal(0, 1, n)
Y = 0.8*X_pre + 1.0*treat + eps            # alpha=0.8, efecto=1
theta = np.cov(Y, X_pre, ddof=1)[0, 1] / np.var(X_pre, ddof=1)
Y_cuped = Y - theta*(X_pre - X_pre.mean())
print(f"theta        = {theta:.3f}  (~ alpha=0.8)")
print(f"Var(Y)       = {np.var(Y, ddof=1):.3f}")
print(f"Var(Y_cuped) = {np.var(Y_cuped, ddof=1):.3f}")
eff_plain = Y[treat==1].mean() - Y[treat==0].mean()
eff_cuped = Y_cuped[treat==1].mean() - Y_cuped[treat==0].mean()
print(f"efecto sin CUPED = {eff_plain:.3f} | con CUPED = {eff_cuped:.3f} (ambos ~1, insesgado)")
assert np.var(Y_cuped, ddof=1) < np.var(Y, ddof=1)

### Ejercicio 2 — Variance reduction
Con `ρ=0.7` la reducción teórica es `1-ρ² = 0.51`. Verificar por simulación.

In [ ]:
rng = np.random.default_rng(42)
n, rho = 50_000, 0.7
X = rng.normal(0, 1, n)
Y = rho*X + np.sqrt(1-rho**2)*rng.normal(0, 1, n)   # corr(X,Y)=rho
theta = np.cov(Y, X, ddof=1)[0, 1] / np.var(X, ddof=1)
Y_cuped = Y - theta*(X - X.mean())
ratio = np.var(Y_cuped, ddof=1) / np.var(Y, ddof=1)
print(f"reduccion teorica (1-rho^2) = {1-rho**2:.3f}")
print(f"var ratio empirico          = {ratio:.3f}")
print(f"reduccion empirica          = {1-ratio:.3f}  (~0.49-0.51 esperado)")
assert abs(ratio - (1-rho**2)) < 0.03

### Ejercicio 3 — Peeking inflado
Bajo `H0`, 2000 experimentos con 5 miradas naive. El % de rechazos sube muy por encima de 0.05.

In [ ]:
rng = np.random.default_rng(42)
reps, n_max = 2000, 5000
looks = np.linspace(1000, n_max, 5).astype(int)
rejects = 0
for _ in range(reps):
    a = rng.normal(0, 1, n_max)
    b = rng.normal(0, 1, n_max)
    rejects += any(stats.ttest_ind(a[:t], b[:t]).pvalue < 0.05 for t in looks)
alpha_naive = rejects/reps
print(f"alpha con 5 miradas naive = {alpha_naive:.3f}  (esperado ~0.13-0.18)")
assert 0.10 < alpha_naive < 0.25

### Ejercicio 4 — O'Brien-Fleming boundaries
Calibración Monte Carlo de las fronteras GST `b_k = C / √t_k`: gastan poco α al principio y controlan el error tipo I global en 0.05, a diferencia de usar 1.96 en cada mirada.

In [ ]:
rng = np.random.default_rng(42)
K = 5
tfrac = np.arange(1, K+1)/K                 # fracciones de informacion
sims = 200_000
inc = rng.normal(0, np.sqrt(1/K), size=(sims, K))
W = np.cumsum(inc, axis=1)                    # movimiento browniano en t_k
Z = W / np.sqrt(tfrac)                         # estadisticos Z estandarizados

def type1(C):
    bound = C / np.sqrt(tfrac)                 # forma O'Brien-Fleming: b_k = C/sqrt(t_k)
    return np.mean(np.any(np.abs(Z) > bound, axis=1))

lo, hi = 1.5, 3.5                              # biseccion para C con type-I global = 0.05
for _ in range(40):
    C = (lo + hi)/2
    if type1(C) > 0.05: lo = C
    else: hi = C
obf = C/np.sqrt(tfrac)
print(f"C O-Brien-Fleming      = {C:.3f}")
print(f"boundaries por mirada  = {np.round(obf, 3)}")
print(f"type-I global (OBF)    = {type1(C):.3f}")
print(f"type-I naive (1.96)    = {np.mean(np.any(np.abs(Z) > 1.96, axis=1)):.3f}  <- inflado")
assert abs(type1(C) - 0.05) < 0.01

### Ejercicio 5 — Always-valid CI (confidence sequence)
`confseq` no está instalada → implementamos la *normal mixture confidence sequence* (Howard et al. 2021) desde cero. Es válida bajo **cualquier** tiempo de parada: cubre el parámetro de forma uniforme en el tiempo.

In [ ]:
rng = np.random.default_rng(42)
n, sigma, mu_true, alpha, rho = 5000, 1.0, 0.0, 0.05, 1.0
t = np.arange(1, n+1)
# radio de la confidence sequence (normal mixture, sub-gaussiano) -- depende solo de t
radius = np.sqrt(2*sigma**2*(t*rho**2 + 1)/(t**2*rho**2) * np.log(np.sqrt(t*rho**2 + 1)/alpha))

data = rng.normal(mu_true, sigma, n)          # una realizacion
cummean = np.cumsum(data)/t
covers_stream = np.all(np.abs(cummean - mu_true) <= radius)
z = stats.norm.ppf(1-alpha/2)
print(f"radio always-valid (t=5000) = {radius[-1]:.4f}  vs IC fijo = {z*sigma/np.sqrt(5000):.4f}")
print(f"cubre mu en TODO t (1 stream) = {covers_stream}")

def cover(seed):                               # cobertura uniforme empirica
    d = np.random.default_rng(seed).normal(mu_true, sigma, n)
    return np.all(np.abs(np.cumsum(d)/t - mu_true) <= radius)
cov = np.mean([cover(s) for s in range(500)])
print(f"cobertura uniforme empirica = {cov:.3f}  (>= 0.95 esperado)")
assert cov >= 0.93